# 🚗 Estadísticas de Precios de Autos en Chile
### Análisis avanzado del mercado automotriz chileno · Datos en tiempo real de MercadoLibre Chile

---
**Qué incluye este análisis:**
- 📊 Distribución de precios por marca, modelo y año
- 📉 Curva de depreciación por marca
- 🔍 Correlación precio vs kilometraje
- 🏷️ Comparativa nuevos vs usados
- ⛽ Análisis por tipo de combustible y transmisión
- 🏆 Ranking de marcas y modelos más económicos/caros
- 💡 Resumen ejecutivo con insights clave

In [ ]:
# Instalar dependencias
!pip install requests pandas plotly numpy tqdm -q

In [ ]:
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from tqdm.notebook import tqdm
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:,.0f}'.format)
print(f'✅ Librerías cargadas | Análisis iniciado: {datetime.now().strftime("%d/%m/%Y %H:%M")}')

## ⚙️ Configuración

In [ ]:
# ─── Configuración general ───────────────────────────────────────────────────
BASE_URL        = "https://api.mercadolibre.com"
SITE_ID         = "MLC"          # Chile en MercadoLibre
CATEGORY_AUTOS  = "MLC1744"      # Autos y Camionetas

# Marcas a analizar (ajustable)
MARCAS = [
    "Toyota", "Chevrolet", "Hyundai", "Kia", "Suzuki",
    "Nissan", "Ford", "Volkswagen", "Honda", "Mazda",
    "Mitsubishi", "Subaru", "Peugeot", "Renault", "Citroën"
]

MAX_POR_MARCA = 100   # ↑ para más datos (máx ~200), ↓ para rapidez
DELAY_SEG     = 0.3   # pausa entre requests (cortesía al API)

# Paleta de colores corporativa
COLORS = px.colors.qualitative.Bold
TEMPLATE = "plotly_white"

print(f'📋 Marcas configuradas: {len(MARCAS)} | Items máx por marca: {MAX_POR_MARCA}')

## 📡 Recolección de Datos

In [ ]:
def fetch_cars(brand: str, limit: int = 50, offset: int = 0) -> list:
    """Consulta el API de MercadoLibre Chile para una marca específica."""
    url = f"{BASE_URL}/sites/{SITE_ID}/search"
    params = {
        "category": CATEGORY_AUTOS,
        "q": brand,
        "limit": min(limit, 50),
        "offset": offset,
    }
    try:
        resp = requests.get(url, params=params, timeout=12)
        resp.raise_for_status()
        return resp.json().get("results", [])
    except Exception as e:
        print(f"  ⚠️  Error al obtener {brand}: {e}")
        return []


def get_attr(attributes: list, attr_id: str):
    """Extrae el valor de un atributo por su ID."""
    for a in attributes:
        if a.get("id") == attr_id:
            return a.get("value_name")
    return None


def parse_item(item: dict) -> dict:
    """Convierte un resultado crudo en un dict estructurado."""
    attrs = item.get("attributes", [])
    return {
        "id":          item.get("id"),
        "titulo":      item.get("title"),
        "precio":      item.get("price"),
        "moneda":      item.get("currency_id"),
        "condicion":   item.get("condition"),
        "marca":       get_attr(attrs, "BRAND"),
        "modelo":      get_attr(attrs, "MODEL"),
        "anio":        get_attr(attrs, "VEHICLE_YEAR"),
        "kilometraje": get_attr(attrs, "VEHICLE_MILEAGE"),
        "combustible": get_attr(attrs, "FUEL_TYPE"),
        "transmision": get_attr(attrs, "TRANSMISSION"),
        "url":         item.get("permalink"),
    }


def get_usd_clp() -> float:
    """Obtiene tipo de cambio USD/CLP desde el API de MercadoLibre."""
    try:
        url = f"{BASE_URL}/currency_conversions/search?from=USD&to=CLP"
        data = requests.get(url, timeout=8).json()
        return data.get("ratio", 950)
    except:
        return 950   # fallback


print('✅ Funciones de scraping definidas')

In [ ]:
USD_CLP = get_usd_clp()
print(f'💱 Tipo de cambio: 1 USD = {USD_CLP:,.0f} CLP')

raw_listings = []

for marca in tqdm(MARCAS, desc="Descargando marcas"):
    collected = 0
    offset = 0
    while collected < MAX_POR_MARCA:
        batch = fetch_cars(marca, limit=50, offset=offset)
        if not batch:
            break
        for item in batch:
            raw_listings.append(parse_item(item))
        collected += len(batch)
        offset += 50
        if len(batch) < 50:
            break
        time.sleep(DELAY_SEG)

print(f'\n✅ Total avisos recolectados: {len(raw_listings):,}')

## 🧹 Limpieza y Transformación de Datos

In [ ]:
df_raw = pd.DataFrame(raw_listings).drop_duplicates(subset="id")

def parse_km(val):
    if pd.isna(val):
        return np.nan
    cleaned = str(val).replace(" km", "").replace("km", "").replace(".", "").replace(",", "").strip()
    try:
        return int(cleaned)
    except:
        return np.nan

df = df_raw.copy()

# Convertir precios a CLP
df["precio_clp"] = df.apply(
    lambda r: r["precio"] * USD_CLP if r["moneda"] == "USD" else r["precio"], axis=1
)
df["precio_m"]   = df["precio_clp"] / 1_000_000   # en millones de CLP
df["precio_usd"] = df["precio_clp"] / USD_CLP

# Campos numéricos
df["km"]         = df["kilometraje"].apply(parse_km)
df["anio_num"]   = pd.to_numeric(df["anio"], errors="coerce")
df["antiguedad"] = datetime.now().year - df["anio_num"]

# Etiquetas en español
df["condicion_es"] = df["condicion"].map(
    {"new": "Nuevo", "used": "Usado", "not_specified": "No especificado"}
).fillna("No especificado")

# Filtros de calidad
PRECIO_MIN, PRECIO_MAX = 1, 500    # millones CLP
ANO_MIN = 1990

df = df[
    df["precio_m"].between(PRECIO_MIN, PRECIO_MAX) &
    df["marca"].notna() &
    (df["anio_num"].isna() | (df["anio_num"] >= ANO_MIN))
].reset_index(drop=True)

print(f'📊 Registros válidos: {len(df):,} (de {len(df_raw):,} originales)')
print(f'   Marcas: {df["marca"].nunique()} | Modelos únicos: {df["modelo"].nunique()}')
print(f'   Rango de precios: {df["precio_m"].min():.1f}M – {df["precio_m"].max():.1f}M CLP')
df[["titulo", "marca", "modelo", "anio_num", "km", "precio_m", "condicion_es"]].head(8)

## 📊 Estadísticas Generales

In [ ]:
resumen = df.groupby("marca")["precio_m"].agg(
    Avisos="count",
    Mínimo="min",
    Mediana="median",
    Promedio="mean",
    Máximo="max",
    Std="std"
).round(1).sort_values("Mediana")

resumen.index.name = "Marca"
print("💰 Precios en millones de CLP")
resumen.style \
    .background_gradient(subset=["Mediana", "Promedio"], cmap="RdYlGn_r") \
    .format("{:.1f}", subset=["Mínimo","Mediana","Promedio","Máximo","Std"]) \
    .set_caption("Resumen de precios por marca (millones CLP)")

In [ ]:
# KPIs globales
total    = len(df)
usados   = (df["condicion_es"] == "Usado").sum()
nuevos   = (df["condicion_es"] == "Nuevo").sum()
med_g    = df["precio_m"].median()
prom_g   = df["precio_m"].mean()
med_km   = df["km"].median()

fig = go.Figure()
kpis = [
    (f"{total:,}",          "Total avisos",       "#2196F3"),
    (f"{nuevos:,}",         "Nuevos",              "#4CAF50"),
    (f"{usados:,}",         "Usados",              "#FF9800"),
    (f"${med_g:.1f}M",      "Precio mediano",      "#9C27B0"),
    (f"${prom_g:.1f}M",     "Precio promedio",     "#E91E63"),
    (f"{med_km/1000:.0f}k", "Km medianos (usado)", "#00BCD4"),
]

for i, (val, label, color) in enumerate(kpis):
    fig.add_trace(go.Indicator(
        mode="number",
        value=None,
        title={"text": f"<b style='font-size:22px;color:{color}'>{val}</b><br><span style='font-size:13px;color:gray'>{label}</span>"},
        domain={"row": 0, "column": i},
    ))

fig.update_layout(
    grid={"rows": 1, "columns": len(kpis)},
    height=150,
    margin=dict(t=20, b=0),
    paper_bgcolor="#f8f9fa",
    title_text="📈 KPIs del Mercado Automotriz Chileno",
    title_x=0.01,
)
fig.show()

## 📦 Distribución de Precios

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Distribución general de precios", "Precios por condición del vehículo"],
)

# Histograma general
fig.add_trace(
    go.Histogram(
        x=df["precio_m"],
        nbinsx=40,
        marker_color="#2196F3",
        name="Todos",
        opacity=0.8,
    ), row=1, col=1
)
fig.add_vline(x=df["precio_m"].median(), line_dash="dash", line_color="red",
              annotation_text=f"Mediana: ${df['precio_m'].median():.1f}M",
              annotation_position="top right", row=1, col=1)

# Box plot por condición
for cond, color in [("Nuevo", "#4CAF50"), ("Usado", "#FF9800")]:
    subset = df[df["condicion_es"] == cond]["precio_m"]
    if len(subset):
        fig.add_trace(
            go.Box(
                y=subset,
                name=cond,
                marker_color=color,
                boxmean=True,
                jitter=0.3,
                pointpos=-1.5,
                boxpoints="outliers",
            ), row=1, col=2
        )

fig.update_xaxes(title_text="Precio (millones CLP)", row=1, col=1)
fig.update_yaxes(title_text="Precio (millones CLP)", row=1, col=2)
fig.update_layout(
    height=420,
    template=TEMPLATE,
    title_text="💰 Distribución de Precios",
    showlegend=True,
)
fig.show()

## 🏷️ Análisis por Marca

In [ ]:
# Box plot de precios por marca (ordenado por mediana)
orden = df.groupby("marca")["precio_m"].median().sort_values().index.tolist()

fig = px.box(
    df, x="marca", y="precio_m",
    category_orders={"marca": orden},
    color="marca", color_discrete_sequence=COLORS,
    points="outliers",
    labels={"marca": "Marca", "precio_m": "Precio (millones CLP)"},
    title="🏷️ Distribución de Precios por Marca",
    template=TEMPLATE,
    height=480,
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Mediana de precio por marca — barras horizontales con conteo de avisos
brand_stats = df.groupby("marca").agg(
    mediana=("precio_m", "median"),
    avisos=("id",  "count")
).sort_values("mediana")

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Precio mediano por marca (millones CLP)", "Total de avisos por marca"
])

fig.add_trace(go.Bar(
    y=brand_stats.index,
    x=brand_stats["mediana"],
    orientation="h",
    marker_color="#2196F3",
    text=brand_stats["mediana"].round(1).astype(str) + "M",
    textposition="outside",
    name="Precio mediano",
), row=1, col=1)

fig.add_trace(go.Bar(
    y=brand_stats.index,
    x=brand_stats["avisos"],
    orientation="h",
    marker_color="#4CAF50",
    text=brand_stats["avisos"],
    textposition="outside",
    name="Avisos",
), row=1, col=2)

fig.update_layout(
    height=480,
    template=TEMPLATE,
    title_text="📊 Rankings por Marca",
    showlegend=False,
    margin=dict(l=120)
)
fig.show()

## 📉 Curva de Depreciación

In [ ]:
df_dep = df.dropna(subset=["anio_num"]).copy()
df_dep = df_dep[df_dep["anio_num"] >= 2005]

# Mediana de precio por año para cada marca (top 8 por avisos)
top_marcas = df["marca"].value_counts().head(8).index.tolist()
dep_data = (
    df_dep[df_dep["marca"].isin(top_marcas)]
    .groupby(["marca", "anio_num"])["precio_m"]
    .median()
    .reset_index()
    .rename(columns={"anio_num": "Año", "precio_m": "Precio mediano (M CLP)"})
)

fig = px.line(
    dep_data, x="Año", y="Precio mediano (M CLP)", color="marca",
    markers=True,
    labels={"marca": "Marca"},
    title="📉 Curva de Depreciación por Marca (precio mediano por año de fabricación)",
    color_discrete_sequence=COLORS,
    template=TEMPLATE,
    height=480,
)
fig.update_traces(line=dict(width=2.5))
fig.update_xaxes(title_text="Año del vehículo")
fig.show()

In [ ]:
# Heatmap: marca vs año → precio mediano
df_heat = (
    df_dep[df_dep["marca"].isin(top_marcas) & (df_dep["anio_num"] >= 2010)]
    .groupby(["marca", "anio_num"])["precio_m"]
    .median()
    .unstack("anio_num")
    .round(1)
)

fig = go.Figure(go.Heatmap(
    z=df_heat.values,
    x=[str(int(c)) for c in df_heat.columns],
    y=df_heat.index.tolist(),
    colorscale="RdYlGn",
    colorbar_title="Precio (M CLP)",
    text=df_heat.values.round(1),
    texttemplate="%{text}M",
    hoverongaps=False,
))
fig.update_layout(
    title_text="🗺️ Mapa de Calor — Precio Mediano (M CLP) por Marca y Año",
    xaxis_title="Año del vehículo",
    template=TEMPLATE,
    height=420,
)
fig.show()

## 🛣️ Precio vs Kilometraje

In [ ]:
df_km = df.dropna(subset=["km"]).copy()
df_km = df_km[(df_km["km"] > 100) & (df_km["km"] < 400_000)]
df_km = df_km[df_km["marca"].isin(top_marcas)]

fig = px.scatter(
    df_km, x="km", y="precio_m", color="marca",
    opacity=0.6, size_max=8,
    trendline="lowess",
    trendline_scope="overall",
    labels={"km": "Kilometraje", "precio_m": "Precio (M CLP)", "marca": "Marca"},
    title="🛣️ Precio vs Kilometraje (con línea de tendencia)",
    color_discrete_sequence=COLORS,
    template=TEMPLATE,
    height=480,
)
fig.update_traces(marker=dict(size=5))
fig.show()

In [ ]:
# Precio por rango de km
bins   = [0, 20_000, 50_000, 100_000, 150_000, 200_000, 400_000]
labels = ["0–20k", "20–50k", "50–100k", "100–150k", "150–200k", "200k+"]

df_km["rango_km"] = pd.cut(df_km["km"], bins=bins, labels=labels)

km_stats = (
    df_km.groupby("rango_km", observed=True)["precio_m"]
    .agg(["median", "mean", "count"])
    .rename(columns={"median": "Mediana", "mean": "Promedio", "count": "Avisos"})
    .reset_index()
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=km_stats["rango_km"].astype(str), y=km_stats["Mediana"],
           name="Precio mediano", marker_color="#2196F3",
           text=km_stats["Mediana"].round(1).astype(str)+"M",
           textposition="outside"),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=km_stats["rango_km"].astype(str), y=km_stats["Avisos"],
               name="Cantidad avisos", mode="lines+markers",
               marker_color="#FF9800", line_dash="dot"),
    secondary_y=True,
)
fig.update_yaxes(title_text="Precio mediano (M CLP)", secondary_y=False)
fig.update_yaxes(title_text="Cantidad de avisos", secondary_y=True)
fig.update_layout(
    title_text="📊 Precio Mediano por Rango de Kilometraje",
    xaxis_title="Rango de km",
    template=TEMPLATE, height=420,
)
fig.show()

## ⛽ Análisis por Combustible y Transmisión

In [ ]:
df_comb = df.dropna(subset=["combustible"]).copy()
df_trans = df.dropna(subset=["transmision"]).copy()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Precio mediano por tipo de combustible",
                    "Precio mediano por transmisión"],
    specs=[[{"type": "bar"}, {"type": "bar"}]]
)

# Combustible
comb_med = (df_comb.groupby("combustible")["precio_m"]
            .agg(["median", "count"]).sort_values("median", ascending=False))
comb_med = comb_med[comb_med["count"] >= 3]

fig.add_trace(go.Bar(
    x=comb_med.index, y=comb_med["median"],
    marker_color=[COLORS[i % len(COLORS)] for i in range(len(comb_med))],
    text=comb_med["median"].round(1).astype(str)+"M",
    textposition="outside", name="",
), row=1, col=1)

# Transmisión
trans_med = (df_trans.groupby("transmision")["precio_m"]
             .agg(["median", "count"]).sort_values("median", ascending=False))
trans_med = trans_med[trans_med["count"] >= 3]

fig.add_trace(go.Bar(
    x=trans_med.index, y=trans_med["median"],
    marker_color=[COLORS[i % len(COLORS)] for i in range(len(trans_med))],
    text=trans_med["median"].round(1).astype(str)+"M",
    textposition="outside", name="",
), row=1, col=2)

fig.update_yaxes(title_text="Precio mediano (M CLP)")
fig.update_layout(
    height=420,
    template=TEMPLATE,
    title_text="⛽ Precio por Combustible y Transmisión",
    showlegend=False,
)
fig.show()

## 🏆 Top Modelos

In [ ]:
df_mod = df.dropna(subset=["modelo"]).copy()
df_mod["marca_modelo"] = df_mod["marca"] + " " + df_mod["modelo"]

mod_stats = (
    df_mod.groupby("marca_modelo")
    .agg(avisos=("id", "count"), mediana=("precio_m", "median"))
    .reset_index()
)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Top 15 modelos MÁS OFERTADOS",
                    "Top 15 modelos MÁS ECONÓMICOS (mediana)"])

# Más ofertados
top_of = mod_stats.nlargest(15, "avisos")
fig.add_trace(go.Bar(
    y=top_of["marca_modelo"], x=top_of["avisos"],
    orientation="h", marker_color="#2196F3",
    text=top_of["avisos"], textposition="outside",
), row=1, col=1)

# Más económicos (con al menos 3 avisos)
top_eco = mod_stats[mod_stats["avisos"] >= 3].nsmallest(15, "mediana")
fig.add_trace(go.Bar(
    y=top_eco["marca_modelo"], x=top_eco["mediana"],
    orientation="h", marker_color="#4CAF50",
    text=top_eco["mediana"].round(1).astype(str)+"M", textposition="outside",
), row=1, col=2)

fig.update_layout(
    height=500,
    template=TEMPLATE,
    title_text="🏆 Top Modelos del Mercado Chileno",
    showlegend=False,
    margin=dict(l=200)
)
fig.show()

## 🔬 Análisis Avanzado de Precios

In [ ]:
# Sunburst: condición → marca → precio mediano
df_sb = (
    df[df["marca"].isin(top_marcas)]
    .groupby(["condicion_es", "marca"])["precio_m"]
    .agg(["median", "count"])
    .reset_index()
    .rename(columns={"median": "precio_mediano", "count": "avisos"})
)

fig = px.sunburst(
    df_sb, path=["condicion_es", "marca"],
    values="avisos",
    color="precio_mediano",
    color_continuous_scale="Blues",
    title="🌐 Sunburst: Condición → Marca (tamaño = avisos, color = precio mediano)",
    height=520,
)
fig.update_layout(template=TEMPLATE, coloraxis_colorbar_title="Precio (M)")
fig.show()

In [ ]:
# Violin plot: distribución de precios por marca (usados)
df_v = df[(df["condicion_es"] == "Usado") & df["marca"].isin(top_marcas)]

fig = px.violin(
    df_v, x="marca", y="precio_m",
    color="marca", color_discrete_sequence=COLORS,
    box=True, points="outliers",
    labels={"marca": "Marca", "precio_m": "Precio (M CLP)"},
    title="🎻 Distribución de Precios — Autos USADOS por Marca",
    template=TEMPLATE,
    height=480,
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Bubble chart: marca vs antigüedad promedio vs precio mediano (tamaño = avisos)
df_bub = (
    df.groupby("marca")
    .agg(
        precio_mediano=("precio_m",   "median"),
        antiguedad_prom=("antiguedad", "mean"),
        avisos=("id", "count")
    )
    .reset_index()
    .dropna()
)

fig = px.scatter(
    df_bub, x="antiguedad_prom", y="precio_mediano",
    size="avisos", color="marca",
    text="marca", size_max=55,
    labels={
        "antiguedad_prom": "Antigüedad promedio (años)",
        "precio_mediano":  "Precio mediano (M CLP)",
        "marca": "Marca",
    },
    title="🫧 Bubble Chart: Precio Mediano vs Antigüedad vs Volumen de Avisos",
    color_discrete_sequence=COLORS,
    template=TEMPLATE,
    height=520,
)
fig.update_traces(textposition="top center")
fig.show()

In [ ]:
# Percentiles de precio por marca
percentiles = [10, 25, 50, 75, 90]
perc_data = (
    df[df["marca"].isin(top_marcas)]
    .groupby("marca")["precio_m"]
    .quantile([p/100 for p in percentiles])
    .unstack()
)
perc_data.columns = [f"P{p}" for p in percentiles]
perc_data = perc_data.sort_values("P50").round(1)

fig = go.Figure()
color_map = {"P10": "#b3cde0", "P25": "#6497b1", "P50": "#03396c", "P75": "#FF8C00", "P90": "#CC0000"}
for p_col in ["P10", "P25", "P50", "P75", "P90"]:
    fig.add_trace(go.Bar(
        name=p_col,
        x=perc_data.index,
        y=perc_data[p_col],
        marker_color=color_map[p_col],
    ))

fig.update_layout(
    barmode="group",
    title_text="📐 Percentiles de Precio por Marca (millones CLP)",
    xaxis_title="Marca",
    yaxis_title="Precio (M CLP)",
    template=TEMPLATE,
    height=460,
    legend_title="Percentil",
)
fig.show()

## 💡 Resumen Ejecutivo

In [ ]:
marca_mas_barata = resumen["Mediana"].idxmin()
marca_mas_cara   = resumen["Mediana"].idxmax()
marca_mas_avisos = df["marca"].value_counts().idxmax()
modelo_mas_comun = df_mod["marca_modelo"].value_counts().idxmax()

dep_ratio = None
if len(df_dep) > 0:
    df_dep_g = df_dep.groupby("anio_num")["precio_m"].median()
    if len(df_dep_g) >= 2:
        oldest_med = df_dep_g.sort_index().iloc[0]
        newest_med = df_dep_g.sort_index().iloc[-1]
        dep_ratio  = ((newest_med - oldest_med) / newest_med) * 100

insights = [
    f"📊 Se analizaron <b>{len(df):,}</b> avisos de <b>{df['marca'].nunique()}</b> marcas distintas.",
    f"💰 El precio mediano del mercado es <b>${df['precio_m'].median():.1f} millones CLP</b>.",
    f"🟢 Marca más económica (mediana): <b>{marca_mas_barata}</b> — ${resumen.loc[marca_mas_barata,'Mediana']:.1f}M.",
    f"🔴 Marca más cara (mediana): <b>{marca_mas_cara}</b> — ${resumen.loc[marca_mas_cara,'Mediana']:.1f}M.",
    f"📈 Marca con más oferta: <b>{marca_mas_avisos}</b> con {df['marca'].value_counts()[marca_mas_avisos]:,} avisos.",
    f"🏆 Modelo más ofertado: <b>{modelo_mas_comun}</b>.",
]

if dep_ratio:
    insights.append(f"📉 Los autos pierden aproximadamente <b>{dep_ratio:.0f}%</b> de valor desde 0 km hasta el modelo más antiguo en el mercado.")

nuevos_pct = (df['condicion_es'] == 'Nuevo').mean() * 100
insights.append(f"🆕 <b>{nuevos_pct:.1f}%</b> de los avisos corresponde a autos nuevos, {100-nuevos_pct:.1f}% a usados.")

html = "<div style='font-family:Arial,sans-serif;background:#f0f4f8;padding:20px;border-radius:10px;'>"
html += "<h2 style='color:#1a237e'>💡 Resumen Ejecutivo del Mercado Automotriz Chileno</h2>"
html += "<ul style='line-height:2.0'>"
for i in insights:
    html += f"<li style='font-size:15px'>{i}</li>"
html += "</ul>"
html += f"<p style='color:gray;font-size:12px'>Fuente: MercadoLibre Chile · Datos a {datetime.now().strftime('%d/%m/%Y %H:%M')}</p>"
html += "</div>"

from IPython.display import HTML
HTML(html)

In [ ]:
# Exportar datos a CSV
csv_path = "chile_autos_datos.csv"
df.to_csv(csv_path, index=False)
print(f'💾 Datos exportados a: {csv_path} ({len(df):,} filas)')